In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate,
                                    PromptTemplate,
                                    ChatMessagePromptTemplate,
                                    ChatPromptTemplate)
from langchain_ollama import ChatOllama

base_url="http://localhost:11434"
model='llama3.2:3b'


llm=ChatOllama(base_url=base_url,model=model)
llm.invoke('hi')

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-21T13:15:15.259169Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2614029833, 'load_duration': 2077164250, 'prompt_eval_count': 26, 'prompt_eval_duration': 352861833, 'eval_count': 8, 'eval_duration': 182942250, 'message': Message(role='assistant', content='', images=None, tool_calls=None)}, id='run--dbf88fe6-5ee3-4e50-951d-c8d4e3a563e4-0', usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [3]:
# TOOL CREATION

from langchain_core.tools import tool

@tool
def add(a,b):
    """ Add two integer numbers together

    Args: 
    a: First integer
    b: Second integer
    """
    return a+b

@tool
def multiply(a,b):
    """
    Multiply two integer numbers togther
    
    Args:
    a: First integer
    b: Second integer
    """

    return a*b

In [4]:
add.name, add.description, add.args, add.args_schema.schema()

/var/folders/14/997y_1sx7mgg_cnwgfmn62m80000gn/T/ipykernel_22069/2446820333.py:1: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  add.name, add.description, add.args, add.args_schema.schema()


('add',
 'Add two integer numbers together\n\n   Args: \n   a: First integer\n   b: Second integer',
 {'a': {'title': 'A'}, 'b': {'title': 'B'}},
 {'description': 'Add two integer numbers together\n\nArgs: \na: First integer\nb: Second integer',
  'properties': {'a': {'title': 'A'}, 'b': {'title': 'B'}},
  'required': ['a', 'b'],
  'title': 'add',
  'type': 'object'})

In [5]:
add.invoke({'a':1,'b':2})

3

In [6]:
multiply.invoke({'a':3,'b':2})

6

In [7]:
tools = [add, multiply]

llm_with_tools = llm.bind_tools(tools)
llm_with_tools

RunnableBinding(bound=ChatOllama(model='llama3.2:3b', base_url='http://localhost:11434'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'add', 'description': 'Add two integer numbers together\n\n   Args: \n   a: First integer\n   b: Second integer', 'parameters': {'properties': {'a': {}, 'b': {}}, 'required': ['a', 'b'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'Multiply two integer numbers togther\n\nArgs:\na: First integer\nb: Second integer', 'parameters': {'properties': {'a': {}, 'b': {}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [8]:
question = "What is 1 multiplied by 2"
llm_with_tools.invoke(question).tool_calls

[{'name': 'multiply',
  'args': {'a': '1', 'b': '2'},
  'id': 'ddd48d7b-de8d-4cf6-9db0-67f240c7dd27',
  'type': 'tool_call'}]

In [9]:
question = "What is 1 multiplied by 2 and what is 86+7?"
llm_with_tools.invoke(question).tool_calls

[{'name': 'multiply',
  'args': {'a': '1', 'b': '2'},
  'id': 'de473339-d78e-4bc4-904a-1d6610c2e65c',
  'type': 'tool_call'},
 {'name': 'add',
  'args': {'a': '86', 'b': '7'},
  'id': '9861d7b4-d2b1-4776-bcc9-c6ae386d97f8',
  'type': 'tool_call'}]

### Calling inbuilt Tools

In [10]:
! pip install -qU duckduckgo-search wikipedia xmltodict tavily-python 

In [11]:
!pip install -U ddgs

In [12]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("What is today's stock market news?")

"Stock Market Today highlights the latest stock market news and analysis. Updated throughout each session, it also include stock futures. Stay up to date with latest developments, trends, and daily insights on the market . Get a comprehensive look at market movements, international updates, economic indicators, and our expert's analysis. Stock Market Today : The Dow Jones index moved toward fresh highs Friday in a week marking the first Fed rate cut of 2025. Stocks closed mixed after the Fed's much-anticipated monetary policy decision. Find the latest stock market news from every corner of the globe at Reuters.com, your online source for breaking international market and finance news"

In [13]:
!pip install langchain-tavily

In [14]:
from langchain_tavily import TavilySearch

search = TavilySearch(
    max_results=5,
    include_answer=True,
    include_raw_content=True,
    # include_images=False,
    # include_image_descriptions=False,
    search_depth="advanced",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [15]:
question = "What is today's stock market news?"

search.invoke(question)

{'query': "What is today's stock market news?",
 'follow_up_questions': None,
 'answer': "Today's stock market shows gains in the S&P 500, Nasdaq, and Dow. Tech stocks lead, with notable rises in Apple and Google. Small caps also outperform large caps.",
 'images': [],
 'results': [{'url': 'https://tradingeconomics.com/united-states/stock-market',
   'title': 'United States Stock Market Index - Quote - Chart - Historical Data',
   'content': 'options expiry takes place today, though volatility is expected to remain limited. Consumer discretionary, tech and communication services booked gains while the energy sector was the biggest laggard. On the corporate front, Apple shares rose about 1.4% as the iPhone 17 goes on sale. Amazon (0.6%), Tesla (2.1%) and Oracle (2.9%) were also higher and FedEx shares added 1% after its first-quarter results beat expectations. [...] US stocks closed at new highs on Friday, extending the record-setting gains from the prior session as investors digested u

### Wikipedia & PubMed

In [16]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# question = "What is the capital of France?"
question = "What is LLM?"
print(wikipedia.invoke(question))

Page: Large language model
Summary: A large language model (LLM) is a language model trained with self-supervised machine learning on a vast amount of text, designed for natural language processing tasks, especially language generation.
The largest and most capable LLMs are generative pre-trained transformers (GPTs), based on a transformer architecture, which are largely used in generative chatbots such as ChatGPT, Gemini and Claude. LLMs can be fine-tuned for specific tasks or guided by prompt engineering. These models acquire predictive power regarding syntax, semantics, and ontologies inherent in human language corpora, but they also inherit inaccuracies and biases present in the data they are trained on.

Page: Gemini (chatbot)
Summary: Gemini (formerly Bard) is a generative artificial intelligence chatbot developed by Google AI. Based on the large language model (LLM) of the same name, it was launched in February 2024. Its predecessor, Bard, was launched in March 2023 in response 

In [17]:
from langchain_community.tools.pubmed.tool import PubmedQueryRun

search = PubmedQueryRun()

print(search.invoke("Cause of diabetes?"))

PubMed exception: Remote end closed connection without response


### Tool calling with LLM

In [ ]:
from langchain_core.tools import tool

@tool
def wikipedia_search(query):
    """ Search wikipedia for general information
    """
    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wikipedia.invoke(query)


@tool
def pubmed_search(query):
    """ Search pubmed for medical and life science queries
    """
    search = PubmedQueryRun()

    return search.invoke(query)
    

@tool
def tavily_search(query):
    """ Realtime and latest information
    """

    search = TavilySearch(max_results=5,
    include_answer=True,
    include_raw_content=True,
    # include_images=False,
    # include_image_descriptions=False,
    search_depth="advanced",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
    )
    return search.invoke(query)


@tool
def multiply(a,b):
    """
    Multiply two integer numbers togther
    
    Args:
    a: First integer
    b: Second integer
    """

    return a*b

In [19]:
tools=[wikipedia_search, pubmed_search, tavily_search, multiply]

list_of_tools ={tool.name: tool for tool in tools}

In [20]:
list_of_tools

{'wikipedia_search': StructuredTool(name='wikipedia_search', description='Search wikipedia for general information', args_schema=<class 'langchain_core.utils.pydantic.wikipedia_search'>, func=<function wikipedia_search at 0x111664b80>),
 'pubmed_search': StructuredTool(name='pubmed_search', description='Search pubmed for medical and life science queries', args_schema=<class 'langchain_core.utils.pydantic.pubmed_search'>, func=<function pubmed_search at 0x126d47d80>),
 'tavily_search': StructuredTool(name='tavily_search', description='Realtime and latest information', args_schema=<class 'langchain_core.utils.pydantic.tavily_search'>, func=<function tavily_search at 0x126d47ec0>),
 'multiply': StructuredTool(name='multiply', description='Multiply two integer numbers togther\n\nArgs:\na: First integer\nb: Second integer', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x123763ce0>)}

In [21]:
llm_with_tools = llm.bind_tools(tools)


In [22]:
# query = "What is the latest news?"
# query = "What is the stock market news?"
# query = "What is LLM?"
# query = "How to treat lung cancer?"
query = "What is 2*3"

response = llm_with_tools.invoke(query)
print(response.tool_calls)

[{'name': 'multiply', 'args': {'a': '2', 'b': '3'}, 'id': '8744bec7-260f-40fd-8b19-30f58a330511', 'type': 'tool_call'}]


### Generate final result with Tool Calling

In [23]:
from langchain_core.messages import HumanMessage

In [24]:
# query = "What is the latest news?"
# query = "What is the stock market news?"
# query = "What is LLM?"
# query = "How to treat lung cancer?"
query = "What is 2*3"

messages=[HumanMessage(query)]

ai_msg = llm_with_tools.invoke(messages)

messages.append(ai_msg)

In [25]:
messages

[HumanMessage(content='What is 2*3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-21T13:18:26.79195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 462921125, 'load_duration': 34219709, 'prompt_eval_count': 318, 'prompt_eval_duration': 26739375, 'eval_count': 16, 'eval_duration': 401630292, 'message': Message(role='assistant', content='', images=None, tool_calls=[ToolCall(function=Function(name='multiply', arguments={'a': 2, 'b': 3}))])}, id='run--c34239fd-9053-4928-82b4-8b2793860fa8-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'a84c1df4-ef25-458e-b6d1-fc7836702a52', 'type': 'tool_call'}], usage_metadata={'input_tokens': 318, 'output_tokens': 16, 'total_tokens': 334})]

In [26]:
for tool_call in ai_msg.tool_calls:

    name = tool_call['name'].lower()
    selected_tool = list_of_tools[name]
    tool_msg = selected_tool.invoke(tool_call['args'])

    messages.append(tool_msg)

In [27]:
messages

[HumanMessage(content='What is 2*3', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2025-09-21T13:18:26.79195Z', 'done': True, 'done_reason': 'stop', 'total_duration': 462921125, 'load_duration': 34219709, 'prompt_eval_count': 318, 'prompt_eval_duration': 26739375, 'eval_count': 16, 'eval_duration': 401630292, 'message': Message(role='assistant', content='', images=None, tool_calls=[ToolCall(function=Function(name='multiply', arguments={'a': 2, 'b': 3}))])}, id='run--c34239fd-9053-4928-82b4-8b2793860fa8-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, 'id': 'a84c1df4-ef25-458e-b6d1-fc7836702a52', 'type': 'tool_call'}], usage_metadata={'input_tokens': 318, 'output_tokens': 16, 'total_tokens': 334}),
 6]